# V7_0_N04 — Earn the Model’s Complexity

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft. Illustrative data do not constitute official statistics or operational authorization.

## Learning outcomes
Compare transparent baselines with stronger models using time-aware validation, calibration, uncertainty, subgroup checks, and stress tests. Select complexity only when it improves decision-relevant performance.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(704); n=180
t=np.arange(n); seasonal=10*np.sin(2*np.pi*t/12); y=80+.18*t+seasonal+rng.normal(0,4,n)
df=pd.DataFrame({'t':t,'y':y}); train=df.iloc[:144].copy(); test=df.iloc[144:].copy()
print(train.shape,test.shape)

(144, 2) (36, 2)


## 1. Time order is part of the evidence
Random splitting leaks future structure into training. Evaluation must simulate the real forecasting origin and horizon.

In [2]:
test['naive']=train.y.iloc[-1]
season_lookup=train.set_index(train.t%12).groupby(level=0).y.mean()
test['seasonal']=test.t.mod(12).map(season_lookup)
coef=np.polyfit(train.t,train.y,1); test['trend']=np.polyval(coef,test.t)
print(test.head().round(2).to_string(index=False))

  t      y  naive  seasonal  trend
144 101.02  102.0     92.77 104.87
145 119.20  102.0     94.58 105.04
146 111.34  102.0    100.65 105.20
147 129.06  102.0    103.35 105.37
148 110.24  102.0    102.92 105.54


## 2. Use several transparent baselines
Last observation, seasonal history, and trend encode different assumptions. A sophisticated model must outperform relevant baselines, not a deliberately weak comparator.

In [3]:
def metrics(actual,pred):
 e=np.asarray(actual)-np.asarray(pred); return {'MAE':np.abs(e).mean(),'RMSE':np.sqrt((e**2).mean()),'Bias':e.mean()}
score=pd.DataFrame({c:metrics(test.y,test[c]) for c in ['naive','seasonal','trend']}).T
print(score.round(3).to_string())

             MAE    RMSE    Bias
naive      9.325  12.015   8.202
seasonal  17.584  18.339  17.584
trend      7.632   9.149   2.377


## 3. A stronger combined model
This illustrative regression combines trend and seasonal terms. Its value is empirical: it must earn adoption on held-out future data.

In [4]:
X=lambda z:np.column_stack([np.ones(len(z)),z,np.sin(2*np.pi*z/12),np.cos(2*np.pi*z/12)])
beta=np.linalg.lstsq(X(train.t),train.y,rcond=None)[0]; test['combined']=X(test.t)@beta
score.loc['combined']=metrics(test.y,test.combined)
print(score.round(3).sort_values('MAE').to_string())

             MAE    RMSE    Bias
combined   3.929   4.731   1.345
trend      7.632   9.149   2.377
naive      9.325  12.015   8.202
seasonal  17.584  18.339  17.584


## 4. Prediction intervals and empirical coverage
Point forecasts conceal uncertainty. Interval quality requires both coverage and useful width, evaluated on future observations.

In [5]:
resid=train.y-X(train.t)@beta; q=np.quantile(np.abs(resid),.90); test['lower']=test.combined-q; test['upper']=test.combined+q
coverage=((test.y>=test.lower)&(test.y<=test.upper)).mean(); width=(test.upper-test.lower).mean()
print('COVERAGE',round(coverage,3),'MEAN_WIDTH',round(width,3))

COVERAGE 0.833 MEAN_WIDTH 13.756


## 5. Stress testing
Test plausible disruption, missing seasonality, changed reporting delay, and extreme values. A model can be accurate on average yet fragile under policy-relevant conditions.

In [6]:
stress=test.copy(); stress['y_stress']=stress.y; stress.loc[stress.index[-6:],'y_stress']+=18
base=metrics(test.y,test.combined)['MAE']; shocked=metrics(stress.y_stress,stress.combined)['MAE']
print('BASE_MAE',round(base,2),'SHOCK_MAE',round(shocked,2),'DEGRADATION',round(shocked/base,2))

BASE_MAE 3.93 SHOCK_MAE 6.66 DEGRADATION 1.7


## 6. Complexity decision card
Performance is necessary but not sufficient. Consider maintainability, latency, explainability, calibration, subgroup harm, and retirement.

In [7]:
best=score.MAE.idxmin(); card={'selected_candidate':best,'baseline_beaten':best=='combined','interval_coverage':float(coverage),'stress_degradation_ratio':float(shocked/base),'decision':'PILOT WITH MONITORING' if best=='combined' and coverage>=.75 else 'RETAIN BASELINE'}
print(card)

{'selected_candidate': 'combined', 'baseline_beaten': True, 'interval_coverage': 0.8333333333333334, 'stress_degradation_ratio': 1.695448402658268, 'decision': 'PILOT WITH MONITORING'}


## Exercises
1. Add rolling-origin evaluation. 2. Compare MAE and RMSE consequences. 3. Recalibrate the interval. 4. Define a retirement trigger.

## Exact solutions
1. Refit/forecast at successive historical origins and aggregate horizon-specific metrics. 2. RMSE penalizes large errors more; choose according to decision loss rather than habit. 3. Estimate residual quantiles on a calibration window and verify held-out coverage and width. 4. Examples: persistent baseline underperformance, failed coverage/calibration, unacceptable subgroup harm, unmaintainable dependencies, or changed purpose.

In [8]:
assert 'combined' in score.index and test[['lower','upper']].notna().all().all()
assert card['decision'] in {'PILOT WITH MONITORING','RETAIN BASELINE'}
print('V7_0_N04_COMPLETE_EXECUTION_PASS')

V7_0_N04_COMPLETE_EXECUTION_PASS
